In [1]:
import os
import io
import json
import random
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import requests

import torch
from torchvision import transforms
from transformers import RobertaTokenizer

from qdrant_client import QdrantClient
from qdrant_client.http import models

from models.scold.model import LVL
from urllib.parse import quote
import boto3
from botocore.client import Config
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Jina AI Reranker API configuration
JINA_API_KEY = os.getenv("JINA_API_KEY")
JINA_RERANK_URL = "https://api.jina.ai/v1/rerank"

In [ ]:
IMAGES_DIR = "data/plantwild/images"
PROMPTS_JSON_PATH = "data/plantwild/plantwild_prompts.json"
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "plantwild_collection"

# Cloudflare R2 configuration
R2_ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = os.getenv("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = os.getenv("R2_SECRET_ACCESS_KEY")
R2_BUCKET = os.getenv("R2_BUCKET", "thesis-bucket")
R2_PATH_PREFIX = "plantwild/gallery/images"

IMAGE_MICROBATCH = 16          # image forward-pass microbatch to avoid GPU OOM
UPLOAD_BATCH_SIZE = 2048       # Qdrant suggests 1k–10k for 100k–1M scale
UPLOAD_PARALLEL = 4            # set to number of CPU cores you want to use

# -------------------------
# Initialize R2 client
# -------------------------
s3_client = boto3.client(
    's3',
    endpoint_url=f'https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com',
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    config=Config(signature_version='s3v4'),
    region_name='auto'
)

# R2 public URL base using custom domain
R2_PUBLIC_DOMAIN = os.getenv("R2_PUBLIC_DOMAIN", f"https://{R2_BUCKET}.{R2_ACCOUNT_ID}.r2.cloudflarestorage.com")

print(f"R2 client initialized for bucket: {R2_BUCKET}")
print(f"R2 endpoint: https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com")
print(f"Public domain: {R2_PUBLIC_DOMAIN}")

# -------------------------
# Load prompts mapping (class -> list of prompts)
# -------------------------
with open(PROMPTS_JSON_PATH, "r") as f:
    class_prompts = json.load(f)

print(f"Loaded prompts for {len(class_prompts)} classes")

# -------------------------
# Helper function to extract plant name from class name
# -------------------------
def extract_plant_name(class_name: str) -> str:
    """
    Extract the plant name from class names.
    Handles two formats:
    1. 'Apple___Cedar_apple_rust' → 'apple'
    2. 'apple black rot' → 'apple'
    
    Args:
        class_name: Class name from dataset
        
    Returns:
        Plant name in lowercase (e.g., 'apple')
    """
    # Try triple underscore format first (e.g., 'Apple___Cedar_apple_rust')
    if '___' in class_name:
        plant_name = class_name.split('___')[0].strip()
        return plant_name.lower()
    
    # Otherwise use space-separated format (e.g., 'apple black rot')
    # Take only the first word as the plant name
    plant_name = class_name.split()[0].strip()
    return plant_name.lower()

# -------------------------
# Model / encoding
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LVL()
model.load_state_dict(torch.load("models/scold/scold.pt", map_location=device))
model.to(device)
model.eval()

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)

def encode_text(texts: list[str]) -> np.ndarray:
    """Returns float32 array of shape (B, 512)."""
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        emb = model.get_texts_feature(inputs["input_ids"], inputs["attention_mask"])
    return emb.detach().cpu().numpy().astype(np.float32, copy=False)


def encode_images_from_files(image_paths: list[Path], microbatch: int = IMAGE_MICROBATCH) -> np.ndarray:
    """Returns float32 array of shape (B, 512)."""
    out = []
    for j in range(0, len(image_paths), microbatch):
        chunk = image_paths[j : j + microbatch]
        imgs = []
        for img_path in chunk:
            img = Image.open(img_path).convert("RGB")
            imgs.append(transform(img))
        batch = torch.stack(imgs, dim=0).to(device)

        with torch.no_grad():
            emb = model.get_images_features(batch)

        out.append(emb.detach().cpu().numpy())

    return np.concatenate(out, axis=0).astype(np.float32, copy=False)


def upload_image_to_r2(image_path: Path, class_name: str) -> str:
    """
    Upload an image to Cloudflare R2 and return its public URL.
    
    Args:
        image_path: Path to the image file
        class_name: Class name for organizing images
        
    Returns:
        Public URL of the uploaded image
    """
    # Create storage path: plantwild/gallery/images/{class_name}/{filename}
    storage_path = f"{R2_PATH_PREFIX}/{class_name}/{image_path.name}"
    
    # Determine content type
    content_type = "image/jpeg" if image_path.suffix.lower() in ['.jpg', '.jpeg'] else "image/png"
    
    # Upload to R2
    try:
        with open(image_path, "rb") as f:
            s3_client.put_object(
                Bucket=R2_BUCKET,
                Key=storage_path,
                Body=f,
                ContentType=content_type
            )
    except Exception as e:
        print(f"\nError uploading {image_path.name}: {e}")
        raise e
    
    # Construct public URL
    public_url = f"{R2_PUBLIC_DOMAIN}/{storage_path}"
    
    return public_url


def generate_r2_url(class_name: str, filename: str) -> str:
    """
    Generate R2 public URL for an image without uploading.
    Assumes the image has already been uploaded to R2.
    
    Args:
        class_name: Class name for organizing images
        filename: Image filename
        
    Returns:
        Public URL of the image
    """
    # URL-encode the class name and filename to handle spaces and special characters
    encoded_path = f"{R2_PATH_PREFIX}/{quote(class_name)}/{quote(filename)}"
    return f"{R2_PUBLIC_DOMAIN}/{encoded_path}"


def test_url_accessibility(url: str, timeout: int = 5) -> bool:
    """
    Test if a URL is accessible via HTTP HEAD request.
    
    Args:
        url: URL to test
        timeout: Request timeout in seconds
        
    Returns:
        True if URL is accessible (status 200), False otherwise
    """
    try:
        response = requests.head(url, timeout=timeout, allow_redirects=True)
        return response.status_code == 200
    except Exception as e:
        return False


# -------------------------
# Qdrant collection setup
# -------------------------
def reset_collection(client: QdrantClient, name: str) -> None:
    if client.collection_exists(name):
        client.delete_collection(name)

    # Create collection with dense vectors only (image/text)
    client.create_collection(
        collection_name=name,
        vectors_config={
            "text": models.VectorParams(size=512, distance=models.Distance.COSINE, on_disk=True),
            "image": models.VectorParams(size=512, distance=models.Distance.COSINE, on_disk=True),
        }
    )
    
    # Create full-text index on plant_name for case-insensitive filtering with MatchText
    client.create_payload_index(
        collection_name=name,
        field_name="plant_name",
        field_schema=models.TextIndexParams(
            type=models.TextIndexType.TEXT,
            tokenizer=models.TokenizerType.WORD,
            min_token_len=2,
            max_token_len=20,
            lowercase=True
        )
    )
    
    # Create full-text index on label (class name) for case-insensitive filtering
    client.create_payload_index(
        collection_name=name,
        field_name="label",
        field_schema=models.TextIndexParams(
            type=models.TextIndexType.TEXT,
            tokenizer=models.TokenizerType.WORD,
            min_token_len=2,
            max_token_len=20,
            lowercase=True
        )
    )
    
    print(f"✓ Created collection '{name}' with:")
    print(f"  - Dense vectors: text (512d), image (512d)")
    print(f"  - Full-text index on 'plant_name' and 'label'")


# -------------------------
# Upload gallery to R2
# -------------------------
def upload_gallery_to_r2(images_dir: str, prompts_dict: dict) -> dict:
    """
    Upload all images from class folders to Cloudflare R2.
    Returns a mapping of (class_name, filename) -> public_url
    
    Args:
        images_dir: Directory containing class folders with images
        prompts_dict: Dictionary mapping class names to lists of prompts
        
    Returns:
        Dictionary mapping (class_name, filename) tuples to public URLs
    """
    url_mapping = {}
    images_path = Path(images_dir)
    
    # Get all class folders
    class_folders = sorted([d for d in images_path.iterdir() if d.is_dir()])
    
    # Count total images for progress bar
    total_images = 0
    for class_folder in class_folders:
        image_files = list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png"))
        total_images += len(image_files)
    
    print(f"Found {len(class_folders)} classes with {total_images:,} total images")
    print("Starting upload to Cloudflare R2...")
    
    pbar = tqdm(total=total_images, desc="Uploading images", unit="img")
    
    try:
        for class_folder in class_folders:
            class_name = class_folder.name
            
            # Get all image files in this class folder
            image_files = sorted(list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png")))
            
            if not image_files:
                continue
            
            print(f"\nUploading class: {class_name} ({len(image_files)} images)")
            
            for img_path in image_files:
                try:
                    url = upload_image_to_r2(img_path, class_name)
                    url_mapping[(class_name, img_path.name)] = url
                    pbar.update(1)
                except Exception as e:
                    print(f"\nFailed to upload {img_path.name}: {e}")
                    pbar.update(1)
                    continue
    finally:
        pbar.close()
    
    print(f"\nUpload complete! Successfully uploaded {len(url_mapping)} images")
    return url_mapping


# -------------------------
# Iterate through image folders -> PointStruct iterator (using pre-uploaded images)
# -------------------------
def iter_points_from_image_folders(images_dir: str, prompts_dict: dict, batch_size: int = 32):
    """
    Iterate through class folders, load images, generate R2 URLs, and yield PointStruct objects.
    For each image, randomly select a prompt from the class's prompts.
    Extracts plant name from class name and stores it as searchable metadata.
    
    Args:
        images_dir: Directory containing class folders with images
        prompts_dict: Dictionary mapping class names to lists of prompts
        batch_size: Number of images to process at once
    """
    point_id = 0
    images_path = Path(images_dir)
    
    # Get all class folders
    class_folders = sorted([d for d in images_path.iterdir() if d.is_dir()])
    
    # Count total images for progress bar
    total_images = 0
    for class_folder in class_folders:
        image_files = list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png"))
        total_images += len(image_files)
    
    print(f"Found {len(class_folders)} classes with {total_images:,} total images")
    
    pbar = tqdm(total=total_images, desc="Processing images", unit="img")

    try:
        for class_folder in class_folders:
            class_name = class_folder.name
            
            # Extract plant name from class name
            plant_name = extract_plant_name(class_name)
            
            # Get prompts for this class
            prompts = prompts_dict.get(class_name, [f"A plant leaf image of {class_name}"])
            
            # Get all image files in this class folder
            image_files = sorted(list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png")))
            
            if not image_files:
                continue
            
            print(f"\nProcessing class: {class_name} ({len(image_files)} images) - Plant: {plant_name}")
            
            # Process in batches
            for i in range(0, len(image_files), batch_size):
                batch_files = image_files[i:i + batch_size]
                
                # Generate URLs for images (assuming they're already uploaded)
                image_urls = [generate_r2_url(class_name, img_path.name) for img_path in batch_files]
                
                # Generate random captions for each image
                captions = [random.choice(prompts) for _ in batch_files]
                
                # Encode images and text (dense vectors)
                img_vecs = encode_images_from_files(batch_files)  # (B, 512)
                text_vecs = encode_text(captions)                 # (B, 512)
                
                # Create points
                for j, img_path in enumerate(batch_files):
                    payload = {
                        "caption": captions[j],
                        "label": class_name,
                        "plant_name": plant_name,  # Add extracted plant name for filtering
                        "source_id": int(point_id),
                        "filename": img_path.name,
                        "image_url": image_urls[j],
                    }

                    yield models.PointStruct(
                        id=int(point_id),
                        vector={
                            "text": text_vecs[j].tolist(),
                            "image": img_vecs[j].tolist()
                        },
                        payload=payload,
                    )

                    point_id += 1
                    pbar.update(1)
    finally:
        pbar.close()
        print(f"\nTotal points generated: {point_id}")


R2 client initialized for bucket: thesis-bucket
R2 endpoint: https://ece14faeb1cd463c0490748fdbb26c82.r2.cloudflarestorage.com
Public domain: https://thesis-assets.andyathsid.com
Loaded prompts for 92 classes


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# STEP 1: Upload gallery images to Cloudflare R2
# Run this cell first to upload all images to R2

# url_mapping = upload_gallery_to_r2(IMAGES_DIR, class_prompts)

# # # Save the URL mapping for reference (optional)
# print(f"\nSuccessfully uploaded {len(url_mapping)} images to R2")
# print(f"Sample URLs:")
# for i, ((class_name, filename), url) in enumerate(list(url_mapping.items())[:5]):
#     print(f"  {class_name}/{filename}: {url}")


Found 89 classes with 18,275 total images
Starting upload to Cloudflare R2...


Uploading images:   0%|          | 0/18275 [00:00<?, ?img/s]


Uploading class: apple black rot (170 images)

Uploading class: apple leaf (441 images)


KeyboardInterrupt: 

In [ ]:
# # Sanity check: Test plant name extraction
# print("Testing plant name extraction:")
# print("=" * 60)

# test_cases = [
#     "apple black rot",
#     "banana panama disease", 
#     "basil downy mildew",
#     "Apple___Cedar_apple_rust",
#     "Tomato___Bacterial_spot",
#     "grape leaf",
#     "strawberry leaf scorch",
#     "Pepper___bell___Bacterial_spot"
# ]

# for test in test_cases:
#     result = extract_plant_name(test)
#     print(f"  '{test}' → '{result}'")

# print("=" * 60)

In [ ]:
# # STEP 2: Sanity check - Test URL accessibility
# # Test a sample of URLs to ensure they are publicly accessible

# print("Testing URL accessibility...")
# sample_size = min(10, len(url_mapping))
# sample_urls = list(url_mapping.values())[:sample_size]

# accessible_count = 0
# failed_urls = []

# for url in tqdm(sample_urls, desc="Testing URLs"):
#     if test_url_accessibility(url):
#         accessible_count += 1
#     else:
#         failed_urls.append(url)

# print(f"\nAccessibility Test Results:")
# print(f"  Total tested: {sample_size}")
# print(f"  Accessible: {accessible_count}")
# print(f"  Failed: {len(failed_urls)}")

# if failed_urls:
#     print(f"\nFailed URLs:")
#     for url in failed_urls:
#         print(f"  - {url}")
#     print("\nWARNING: Some URLs are not accessible!")
#     print("Please check your R2 bucket's public access settings.")
#     print("You may need to:")
#     print("  1. Enable public access on your R2 bucket")
#     print("  2. Set up a custom domain with R2")
#     print("  3. Update R2_PUBLIC_DOMAIN variable with your actual public domain")
# else:
#     print("\n✅ All tested URLs are accessible!")
#     print("You can proceed to the next step to ingest data to Qdrant.")


In [4]:
# STEP 3: Ingest data to Qdrant with R2 image URLs
# Run this cell ONLY AFTER uploading images to R2 and verifying URL accessibility

client = QdrantClient(
    url=QDRANT_URL, 
    api_key=QDRANT_API_KEY,
    timeout=60,
    prefer_grpc=True
)

reset_collection(client, COLLECTION_NAME)

# Process images from class folders (uses pre-uploaded R2 images)
points_iter = iter_points_from_image_folders(
    IMAGES_DIR, 
    class_prompts, 
    batch_size=32
)

# Upload to Qdrant with batching + parallelism
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=points_iter,
    batch_size=UPLOAD_BATCH_SIZE,
    parallel=UPLOAD_PARALLEL,
    wait=False,
)

print("\n✅ Data ingestion to Qdrant initiated!")
print("Note: Ingestion is running in the background (wait=False)")


✓ Created collection 'plantwild_collection' with:
  - Dense vectors: text (512d), image (512d)
  - Full-text index on 'plant_name' and 'label'
Found 89 classes with 18,275 total images


Processing images:   0%|          | 0/18275 [00:00<?, ?img/s]


Processing class: apple black rot (170 images) - Plant: apple

Processing class: apple leaf (441 images) - Plant: apple

Processing class: apple mosaic virus (178 images) - Plant: apple

Processing class: apple rust (305 images) - Plant: apple

Processing class: apple scab (289 images) - Plant: apple

Processing class: banana leaf (240 images) - Plant: banana

Processing class: banana panama disease (213 images) - Plant: banana

Processing class: basil downy mildew (83 images) - Plant: basil

Processing class: basil leaf (586 images) - Plant: basil

Processing class: bean halo blight (112 images) - Plant: bean

Processing class: bean leaf (255 images) - Plant: bean

Processing class: bean mosaic virus (122 images) - Plant: bean

Processing class: bean rust (162 images) - Plant: bean

Processing class: bell pepper leaf (237 images) - Plant: bell

Processing class: bell pepper leaf spot (114 images) - Plant: bell

Processing class: blueberry leaf (278 images) - Plant: blueberry

Process

d:\Workspace\Repository\thesis\research\image-retrieval-engine\.image-retrieval-venv\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))



Processing class: cabbage alternaria leaf spot (125 images) - Plant: cabbage

Processing class: cabbage leaf (461 images) - Plant: cabbage

Processing class: carrot cavity spot (71 images) - Plant: carrot

Processing class: cauliflower alternaria leaf spot (95 images) - Plant: cauliflower

Processing class: cauliflower leaf (192 images) - Plant: cauliflower

Processing class: celery anthracnose (41 images) - Plant: celery

Processing class: celery early blight (52 images) - Plant: celery

Processing class: celery leaf (209 images) - Plant: celery

Processing class: cherry leaf (283 images) - Plant: cherry

Processing class: cherry leaf spot (227 images) - Plant: cherry

Processing class: cherry powdery mildew (111 images) - Plant: cherry

Processing class: citrus canker (532 images) - Plant: citrus

Processing class: citrus greening disease (234 images) - Plant: citrus

Processing class: coffee leaf (135 images) - Plant: coffee

Processing class: coffee leaf rust (187 images) - Plant:

In [5]:
if client.collection_exists(COLLECTION_NAME):
    print(f"Collection '{COLLECTION_NAME}' exists")
    
    # Get collection info
    info = client.get_collection(COLLECTION_NAME)
    print(f"Total points: {info.points_count}")
    print(f"Vectors: {list(info.config.params.vectors.keys())}")
else:
    print(f"Collection '{COLLECTION_NAME}' does not exist")

Collection 'plantwild_collection' exists
Total points: 18112
Vectors: ['text', 'image']


In [6]:
def cross_modal_search(
    query_text=None, 
    image_path=None, 
    limit=5, 
    search_against="auto"
):
    """
    Unified cross-modal search supporting all 4 modalities:
    1. Text-to-Text: Text query → Caption vectors (semantic text matching)
    2. Text-to-Image: Text query → Image vectors (true cross-modal)
    3. Image-to-Image: Image query → Image vectors (visual similarity)
    4. Image-to-Text: Image query → Caption vectors (cross-modal)
    
    Args:
        query_text: Text query for text-based search
        image_path: Image path for image-based search
        limit: Number of results to return
        search_against: What to search against:
            - "auto": Smart default (text→text, image→image)
            - "text": Search against caption text vectors
            - "image": Search against image vectors
            
    Returns:
        Search results from Qdrant
    """
    if query_text is None and image_path is None:
        raise ValueError("Either query_text or image_path must be provided")
    
    # Determine what to search against
    if search_against == "auto":
        # Default: text queries search text (captions), image queries search images
        if query_text is not None:
            search_against = "text"
        else:
            search_against = "image"
    
    # === TEXT QUERY ===
    if query_text is not None:
        query_vector = encode_text([query_text])[0].tolist()
        
        if search_against == "text":
            # TEXT-TO-TEXT: Compare text query with caption text vectors
            print(f"🔍 Search Mode: Text-to-Text (matching captions)")
            
            # Dense-only search
            search_result = client.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector,
                using="text",
                limit=limit,
                with_payload=True
            )
        
        elif search_against == "image":
            # TEXT-TO-IMAGE: Compare text query with image vectors (CROSS-MODAL!)
            print(f"🔍 Search Mode: Text-to-Image (cross-modal: text→visual content)")
            search_result = client.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector,
                using="image",  # Search image vectors with text encoding!
                limit=limit,
                with_payload=True
            )
        else:
            raise ValueError(f"Invalid search_against: {search_against}. Use 'text' or 'image'")
    
    # === IMAGE QUERY ===
    elif image_path is not None:
        img = Image.open(image_path).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            query_vector = model.get_images_features(img_tensor)[0].cpu().numpy().tolist()
        
        if search_against == "image":
            # IMAGE-TO-IMAGE: Compare image query with image vectors
            print(f"🔍 Search Mode: Image-to-Image (visual similarity)")
            search_result = client.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector,
                using="image",
                limit=limit,
                with_payload=True
            )
        
        elif search_against == "text":
            # IMAGE-TO-TEXT: Compare image query with caption text vectors (CROSS-MODAL!)
            print(f"🔍 Search Mode: Image-to-Text (cross-modal: visual→captions)")
            search_result = client.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector,
                using="text",  # Search text vectors with image encoding!
                limit=limit,
                with_payload=True
            )
        else:
            raise ValueError(f"Invalid search_against: {search_against}. Use 'text' or 'image'")
    
    return search_result


In [7]:
text_query = "Grape vine leaf with small round light spots and brown dead areas."

# Perform search
text_results = cross_modal_search(query_text=text_query, limit=5)

print("Text-to-Image Search Results:")
print("=" * 80)
for i, result in enumerate(text_results.points, 1): 
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption']}")
    print(f"   Filename: {result.payload['filename']}")
    print(f"   Image URL: {result.payload.get('image_url', 'N/A')}")
    print(f"   Source ID: {result.payload['source_id']}")
    print()


🔍 Search Mode: Text-to-Text (matching captions)
Text-to-Image Search Results:
1. Score: 0.8836
   Label: grape black rot
   Caption: Grape black rot appears as small black spots on the fruit, stems, and leaves of grapevines.
   Filename: 76.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/grape%20black%20rot/76.jpg
   Source ID: 10527

2. Score: 0.8836
   Label: grape black rot
   Caption: Grape black rot appears as small black spots on the fruit, stems, and leaves of grapevines.
   Filename: google_0008.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/grape%20black%20rot/google_0008.jpg
   Source ID: 10543

3. Score: 0.8836
   Label: grape black rot
   Caption: Grape black rot appears as small black spots on the fruit, stems, and leaves of grapevines.
   Filename: google_0260.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/grape%20black%20rot/google_0260.jpg
   Source ID: 10655

4. Score: 0.8

In [8]:
# Image-to-Image search
image_query_path = "apple.jpg"

# Perform search
image_results = cross_modal_search(image_path=image_query_path, limit=5)

print("Image-to-Image Search Results:")
print("=" * 80)
for i, result in enumerate(image_results.points, 1): 
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption']}")
    print(f"   Filename: {result.payload['filename']}")
    print(f"   Image URL: {result.payload.get('image_url', 'N/A')}")
    print(f"   Source ID: {result.payload['source_id']}")
    print()


🔍 Search Mode: Image-to-Image (visual similarity)
Image-to-Image Search Results:
1. Score: 0.8839
   Label: apple rust
   Caption: Apple rust appears as bright orange spots or pustules on the leaves, stems, and fruit of apple trees.
   Filename: google_0039.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/apple%20rust/google_0039.jpg
   Source ID: 940

2. Score: 0.8486
   Label: apple rust
   Caption: Apple rust appears as orange or yellow spots on the leaves, often accompanied by dark clusters of spores.
   Filename: google_0007.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/apple%20rust/google_0007.jpg
   Source ID: 915

3. Score: 0.8401
   Label: apple black rot
   Caption: Apple black rot causes circular black lesions with concentric rings and may also cause fruit mummification.
   Filename: google_0000.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/apple%20black%20rot/google_0000.jpg
 

In [9]:
# Image-to-Image search
image_query_path = r"maize.jpg"

# Perform search
image_results = cross_modal_search(image_path=image_query_path, limit=5)

print("Image-to-Image Search Results:")
print("=" * 80)
for i, result in enumerate(image_results.points, 1): 
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption']}")
    print(f"   Filename: {result.payload['filename']}")
    print(f"   Image URL: {result.payload.get('image_url', 'N/A')}")
    print(f"   Source ID: {result.payload['source_id']}")
    print()


🔍 Search Mode: Image-to-Image (visual similarity)
Image-to-Image Search Results:
1. Score: 0.8042
   Label: corn gray leaf spot
   Caption: Corn gray leaf spot appears as small, elliptical, gray lesions with dark borders on the leaves.
   Filename: google_0151.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/corn%20gray%20leaf%20spot/google_0151.jpg
   Source ID: 7334

2. Score: 0.8024
   Label: corn gray leaf spot
   Caption: Corn gray leaf spot appears as dark gray to black lesions with distinct rectangular shapes on the leaves.
   Filename: google_0025.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/corn%20gray%20leaf%20spot/google_0025.jpg
   Source ID: 7252

3. Score: 0.7834
   Label: corn gray leaf spot
   Caption: Corn gray leaf spot appears as long, narrow gray lesions surrounded by dark borders on the leaves.
   Filename: google_0029.jpg
   Image URL: https://thesis-assets.andyathsid.com/plantwild/gallery/images/co

In [10]:
# Test case-insensitive filtering by plant name using MatchText
text_query = "pepper leaf circular dark spots yellow halo damaged edges"

filter_by_plant = models.Filter(
    must=[
        models.FieldCondition(
            key="label",
            match=models.MatchText(text="bell pepper")  # Case-insensitive thanks to full-text index
        )
    ]
)

# Perform filtered search
filtered_results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=encode_text([text_query])[0].tolist(),
    using="text",
    query_filter=filter_by_plant,
    limit=5,
    with_payload=True
)

print("Filtered Search Results:")
print("=" * 80)
for i, result in enumerate(filtered_results.points, 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Plant Name: {result.payload['plant_name']}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption']}")
    print(f"   Filename: {result.payload['filename']}")
    print()


Filtered Search Results:
1. Score: 0.5445
   Plant Name: bell
   Label: bell pepper leaf spot
   Caption: Small circular dark brown spots surrounded by a yellow halo on the leaves of the bell pepper plant.
   Filename: google_0053.jpg

2. Score: 0.5445
   Plant Name: bell
   Label: bell pepper leaf spot
   Caption: Small circular dark brown spots surrounded by a yellow halo on the leaves of the bell pepper plant.
   Filename: google_0172.jpg

3. Score: 0.5445
   Plant Name: bell
   Label: bell pepper leaf spot
   Caption: Small circular dark brown spots surrounded by a yellow halo on the leaves of the bell pepper plant.
   Filename: google_0101.jpg

4. Score: 0.5445
   Plant Name: bell
   Label: bell pepper leaf spot
   Caption: Small circular dark brown spots surrounded by a yellow halo on the leaves of the bell pepper plant.
   Filename: 37.jpg

5. Score: 0.5445
   Plant Name: bell
   Label: bell pepper leaf spot
   Caption: Small circular dark brown spots surrounded by a yellow halo

## Multimodal Reranking with Jina Reranker M0

Use Jina's multimodal reranker ([jina-reranker-m0](https://huggingface.co/jinaai/jina-reranker-m0)) to improve search quality by reranking results based on both text and image understanding.

### Why Reranking?

While our hybrid search (dense SCOLD + sparse SPLADE) provides good initial retrieval, a reranker can significantly improve relevance by:

1. **Multimodal Understanding**: Analyzes both image content and text captions together
2. **Deeper Semantic Analysis**: Goes beyond surface-level matching to understand query intent
3. **Cross-modal Reasoning**: Can reason about relationships between visual and textual information
4. **Better Ranking**: Uses a more sophisticated model (2.4B params) specifically trained for ranking

### Setup

To use the Jina Reranker API, you need to:

1. Get a free API key from [https://jina.ai/reranker](https://jina.ai/reranker)
2. Add it to your `.env` file:
   ```
   JINA_API_KEY=your_api_key_here
   ```

The free tier provides **10 million tokens** - sufficient for experimentation!

In [11]:
def rerank_with_jina(query: str, results, top_n: int = 5, query_type: str = "text") -> dict:
    """
    Rerank search results using Jina's multimodal reranker API.
    
    Args:
        query: Text or image URL query
        results: Qdrant search results with image URLs in payload
        top_n: Number of top results to return
        query_type: "text" or "image" - type of the query
        
    Returns:
        Reranked results with relevance scores
    """
    if not JINA_API_KEY:
        print("Warning: JINA_API_KEY not found. Skipping reranking.")
        return {"results": [{"index": i, "relevance_score": 0} for i in range(len(results.points))]}
    
    # Prepare documents for reranking
    # Jina reranker supports both text and images in documents
    documents = []
    for point in results.points:
        # Use image URL from payload for multimodal reranking
        doc = {
            "image": point.payload.get("image_url", ""),
            "text": point.payload.get("caption", "")
        }
        documents.append(doc)
    
    # Prepare request payload
    payload = {
        "model": "jina-reranker-m0",
        "query": query,
        "documents": documents,
        "top_n": top_n,
        "return_documents": False
    }
    
    # Make API request
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {JINA_API_KEY}"
    }
    
    try:
        response = requests.post(JINA_RERANK_URL, json=payload, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"Error calling Jina Reranker API: {e}")
        if hasattr(e.response, 'text'):
            print(f"Response: {e.response.text}")
        return {"results": [{"index": i, "relevance_score": 0} for i in range(len(results.points))]}


def search_and_rerank(
    query_text=None, 
    image_path=None, 
    limit=20, 
    use_reranker=True,
    rerank_top_n=5,
    query_filter=None,
    search_against="auto"
):
    """
    Perform search and optionally rerank results using Jina's multimodal reranker.
    Supports all 4 search modalities:
    
    1. Text-to-Text: query_text + search_against="text"
    2. Text-to-Image: query_text + search_against="image" (cross-modal)
    3. Image-to-Image: image_path + search_against="image"
    4. Image-to-Text: image_path + search_against="text" (cross-modal)
    
    Args:
        query_text: Text query for search
        image_path: Image path for image-based search
        limit: Number of initial results to retrieve
        use_reranker: Apply Jina multimodal reranker to results
        rerank_top_n: Number of top results after reranking
        query_filter: Qdrant filter for filtering results
        search_against: "auto", "text", or "image" - what vectors to search
        
    Returns:
        Tuple of (initial_results, reranked_results) if use_reranker=True, else just initial_results
    """
    # Determine search mode
    if search_against == "auto":
        if query_text is not None:
            search_against = "text"
        else:
            search_against = "image"
    
    # Step 1: Initial retrieval using cross_modal_search
    if query_text is not None:
        query_vector = encode_text([query_text])[0].tolist()
        
        # Text-to-Text or Text-to-Image (both use dense-only search)
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            using=search_against,
            query_filter=query_filter,
            limit=limit,
            with_payload=True
        )
        
        query_for_rerank = query_text
        query_type = "text"
            
    elif image_path is not None:
        img = Image.open(image_path).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            query_vector = model.get_images_features(img_tensor)[0].cpu().numpy().tolist()
        
        # Image-to-Image or Image-to-Text
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            using=search_against,
            query_filter=query_filter,
            limit=limit,
            with_payload=True
        )
        
        # For image queries, we could use image URL if available
        query_for_rerank = image_path  # In production, this should be an accessible URL
        query_type = "image"
    else:
        raise ValueError("Either query_text or image_path must be provided")
    
    # Step 2: Reranking (optional)
    if use_reranker and len(search_result.points) > 0:
        print(f"\n🔄 Reranking {len(search_result.points)} results using Jina Reranker M0...")
        rerank_response = rerank_with_jina(
            query_for_rerank, 
            search_result, 
            top_n=rerank_top_n,
            query_type=query_type
        )
        
        # Map reranked results back to original points
        reranked_results = []
        for rerank_item in rerank_response.get("results", []):
            idx = rerank_item["index"]
            score = rerank_item["relevance_score"]
            point = search_result.points[idx]
            reranked_results.append({
                "point": point,
                "rerank_score": score,
                "original_score": point.score
            })
        
        return search_result, reranked_results
    
    return search_result, None


In [12]:
# Example: Text query with semantic search + multimodal reranking
query = "grape leaf with black spots and fungal disease"

# Perform search with reranking
initial_results, reranked = search_and_rerank(
    query_text=query,
    limit=20,  # Get more initial candidates
    use_reranker=True,
    rerank_top_n=5
)

print("=" * 80)
print("INITIAL SEARCH RESULTS (Top 5)")
print("=" * 80)
for i, result in enumerate(initial_results.points[:5], 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Plant: {result.payload['plant_name']}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption'][:100]}...")
    print()

if reranked:
    print("\n" + "=" * 80)
    print("RERANKED RESULTS (Jina Reranker M0 - Multimodal)")
    print("=" * 80)
    for i, item in enumerate(reranked, 1):
        point = item["point"]
        print(f"{i}. Rerank Score: {item['rerank_score']:.4f} (Original: {item['original_score']:.4f})")
        print(f"   Plant: {point.payload['plant_name']}")
        print(f"   Label: {point.payload['label']}")
        print(f"   Caption: {point.payload['caption'][:100]}...")
        print(f"   Image URL: {point.payload['image_url']}")
        print()



🔄 Reranking 20 results using Jina Reranker M0...
INITIAL SEARCH RESULTS (Top 5)
1. Score: 0.8241
   Plant: grape
   Label: grape leaf spot
   Caption: Grape leaf spot appears as circular, dark brown lesions with yellow halos on the foliage....

2. Score: 0.8241
   Plant: grape
   Label: grape leaf spot
   Caption: Grape leaf spot appears as circular, dark brown lesions with yellow halos on the foliage....

3. Score: 0.8149
   Plant: grape
   Label: grape black rot
   Caption: Grape black rot appears as dark, necrotic spots on grape berries with concentric rings of black fung...

4. Score: 0.8146
   Plant: grape
   Label: grape black rot
   Caption: Grape black rot appears as dark, necrotic spots on grape berries with concentric rings of black fung...

5. Score: 0.8028
   Plant: grape
   Label: grape black rot
   Caption: Grape black rot appears as dark, shriveled grapes with black fungal spores on the surface....


RERANKED RESULTS (Jina Reranker M0 - Multimodal)
1. Rerank Score: 0.95

In [13]:
# Example: Search with plant filter + multimodal reranking
query = "leaf with brown spots and disease symptoms"

# Filter for apple plants
filter_apple = models.Filter(
    must=[
        models.FieldCondition(
            key="plant_name",
            match=models.MatchText(text="apple")
        )
    ]
)

# Perform filtered search with reranking
initial_results, reranked = search_and_rerank(
    query_text=query,
    limit=15,
    use_reranker=True,
    rerank_top_n=5,
    query_filter=filter_apple
)

print("=" * 80)
print(f"FILTERED SEARCH RESULTS (Plant: apple)")
print("=" * 80)
print(f"Total initial results: {len(initial_results.points)}")
print()

if reranked:
    print("TOP 5 RERANKED RESULTS (Jina Reranker M0):")
    print("=" * 80)
    for i, item in enumerate(reranked, 1):
        point = item["point"]
        print(f"{i}. Rerank Score: {item['rerank_score']:.4f} (↑ from {item['original_score']:.4f})")
        print(f"   Plant: {point.payload['plant_name']}")
        print(f"   Label: {point.payload['label']}")
        print(f"   Caption: {point.payload['caption'][:100]}...")
        print()



🔄 Reranking 15 results using Jina Reranker M0...
FILTERED SEARCH RESULTS (Plant: apple)
Total initial results: 15

TOP 5 RERANKED RESULTS (Jina Reranker M0):
1. Rerank Score: 0.9500 (↑ from 0.4633)
   Plant: apple
   Label: apple leaf
   Caption: The apple leaf appears pale, wilted, and exhibits brown spots indicative of fungal infection....

2. Rerank Score: 0.9500 (↑ from 0.4633)
   Plant: apple
   Label: apple leaf
   Caption: The apple leaf appears pale, wilted, and exhibits brown spots indicative of fungal infection....

3. Rerank Score: 0.9500 (↑ from 0.4633)
   Plant: apple
   Label: apple leaf
   Caption: The apple leaf appears pale, wilted, and exhibits brown spots indicative of fungal infection....

4. Rerank Score: 0.9500 (↑ from 0.4633)
   Plant: apple
   Label: apple leaf
   Caption: The apple leaf appears pale, wilted, and exhibits brown spots indicative of fungal infection....

5. Rerank Score: 0.9500 (↑ from 0.4633)
   Plant: apple
   Label: apple leaf
   Caption: The 

In [14]:
# Comparison: With vs Without Reranking
query = "tomato leaf with yellow edges and wilting"

print("=" * 80)
print("SEARCH WITHOUT RERANKING")
print("=" * 80)

# Without reranking
results_no_rerank, _ = search_and_rerank(
    query_text=query,
    limit=10,
    use_reranker=False
)

for i, result in enumerate(results_no_rerank.points[:5], 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Plant: {result.payload['plant_name']} | Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption'][:80]}...")
    print()

print("\n" + "=" * 80)
print("SEARCH WITH JINA MULTIMODAL RERANKING")
print("=" * 80)

# With reranking
results_with_rerank, reranked = search_and_rerank(
    query_text=query,
    limit=10,
    use_reranker=True,
    rerank_top_n=5
)

if reranked:
    for i, item in enumerate(reranked, 1):
        point = item["point"]
        # Calculate rank change
        original_rank = next((idx for idx, p in enumerate(results_no_rerank.points) if p.id == point.id), -1) + 1
        rank_change = original_rank - i if original_rank > 0 else 0
        
        print(f"{i}. Rerank Score: {item['rerank_score']:.4f}")
        print(f"   Plant: {point.payload['plant_name']} | Label: {point.payload['label']}")
        print(f"   Caption: {point.payload['caption'][:80]}...")
        if rank_change != 0:
            print(f"   📈 Rank change: {'+' if rank_change > 0 else ''}{rank_change} (was #{original_rank})")
        print()


SEARCH WITHOUT RERANKING
1. Score: 0.8093
   Plant: tomato | Label: tomato early blight
   Caption: Tomato early blight presents as dark, concentric spots on leaves and stems, with...

2. Score: 0.8093
   Plant: tomato | Label: tomato early blight
   Caption: Tomato early blight presents as dark, concentric spots on leaves and stems, with...

3. Score: 0.8093
   Plant: tomato | Label: tomato early blight
   Caption: Tomato early blight presents as dark, concentric spots on leaves and stems, with...

4. Score: 0.8093
   Plant: tomato | Label: tomato early blight
   Caption: Tomato early blight presents as dark, concentric spots on leaves and stems, with...

5. Score: 0.8093
   Plant: tomato | Label: tomato early blight
   Caption: Tomato early blight presents as dark, concentric spots on leaves and stems, with...


SEARCH WITH JINA MULTIMODAL RERANKING

🔄 Reranking 10 results using Jina Reranker M0...
1. Rerank Score: 0.9420
   Plant: tomato | Label: tomato early blight
   Caption: Toma

## All 4 Search Modalities Demonstrated

Demonstrating all possible search combinations with SCOLD's cross-modal capabilities.

### Understanding the 4 Search Modes

| Mode | Query Type | Searches Against | Cross-Modal? | Use Case |
|------|-----------|-----------------|--------------|----------|
| **Text-to-Text** | Text | Caption vectors | ❌ No | Find images with similar captions/descriptions |
| **Text-to-Image** | Text | Image vectors | ✅ Yes | Find images that *look like* your text description |
| **Image-to-Image** | Image | Image vectors | ❌ No | Find visually similar images |
| **Image-to-Text** | Image | Caption vectors | ✅ Yes | Find images with captions describing your image |

### Parameters

```python
cross_modal_search(
    query_text="...",          # Text query (for text-based search)
    image_path="...",          # Image path (for image-based search)
    search_against="auto",     # "auto", "text", or "image"
    use_hybrid=True,           # Use hybrid search for text-to-text
    limit=5
)
```

- `search_against="text"`: Search caption text vectors
- `search_against="image"`: Search image vectors (visual content)
- `search_against="auto"`: Smart default (text→text, image→image)

In [ ]:
# 1. TEXT-TO-TEXT: Text query → Caption text vectors
# Find images with captions similar to the query text
query = "tomato leaf with brown spots and early blight disease"

results = cross_modal_search(
    query_text=query,
    search_against="text",  # Search caption text vectors
    limit=5
)

print("Results:")
print("=" * 80)
for i, result in enumerate(results.points, 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Plant: {result.payload['plant_name']}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption']}")
    print()


In [ ]:
# 2. TEXT-TO-IMAGE: Text query → Image vectors (CROSS-MODAL!)
# Find images whose VISUAL CONTENT matches the text description
query = "leaf with orange-brown rusty spots and fungal disease"

results = cross_modal_search(
    query_text=query,
    search_against="image",  # Search image vectors (visual content!)
    limit=5
)

print("Results:")
print("=" * 80)
for i, result in enumerate(results.points, 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Plant: {result.payload['plant_name']}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption']}")
    print(f"   💡 Note: Found by VISUAL similarity, not caption matching!")
    print()


In [ ]:
# 3. IMAGE-TO-IMAGE: Image query → Image vectors
# Find visually similar images
image_query_path = "applerust.jpg"

results = cross_modal_search(
    image_path=image_query_path,
    search_against="image",  # Search image vectors (default for images)
    limit=5
)

print("Results:")
print("=" * 80)
for i, result in enumerate(results.points, 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Plant: {result.payload['plant_name']}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption'][:80]}...")
    print(f"   💡 Note: Found by visual similarity to query image")
    print()


In [ ]:
# 4. IMAGE-TO-TEXT: Image query → Caption text vectors (CROSS-MODAL!)
# Find images whose CAPTIONS describe what's in the query image
image_query_path = "grapeblackrot.jpg"

results = cross_modal_search(
    image_path=image_query_path,
    search_against="text",  # Search caption text vectors with image query!
    limit=5
)

print("Results:")
print("=" * 80)
for i, result in enumerate(results.points, 1):
    print(f"{i}. Score: {result.score:.4f}")
    print(f"   Plant: {result.payload['plant_name']}")
    print(f"   Label: {result.payload['label']}")
    print(f"   Caption: {result.payload['caption']}")
    print(f"   💡 Note: Found images with CAPTIONS that describe the query image")
    print()


## Search Modality Comparison

Compare how different search modes produce different results for the same query.

In [ ]:
# Compare Text-to-Text vs Text-to-Image for the same text query
query = "apple leaf with rust disease and orange spots"

print("=" * 100)
print("TEXT QUERY: " + query)
print("=" * 100)

# Mode 1: Text-to-Text (search captions)
print("\n1️⃣  TEXT-TO-TEXT (Query Text → Caption Vectors)")
print("-" * 100)
print("What it does: Finds images with CAPTIONS that match your text query")
print("-" * 100)

results_t2t = cross_modal_search(
    query_text=query,
    search_against="text",
    limit=3
)

for i, result in enumerate(results_t2t.points, 1):
    print(f"\n{i}. Score: {result.score:.4f} | {result.payload['plant_name']} - {result.payload['label']}")
    print(f"   Caption: {result.payload['caption'][:80]}...")

# Mode 2: Text-to-Image (search visual content)
print("\n\n2️⃣  TEXT-TO-IMAGE (Query Text → Image Vectors) - CROSS-MODAL")
print("-" * 100)
print("What it does: Finds images whose VISUAL CONTENT matches your text description")
print("-" * 100)

results_t2i = cross_modal_search(
    query_text=query,
    search_against="image",  # Cross-modal!
    limit=3
)

for i, result in enumerate(results_t2i.points, 1):
    print(f"\n{i}. Score: {result.score:.4f} | {result.payload['plant_name']} - {result.payload['label']}")
    print(f"   Caption: {result.payload['caption'][:80]}...")
    print(f"   💡 Found by VISUAL match, caption may be different!")

print("\n" + "=" * 100)
print("Notice how results differ based on whether we match captions or visual content!")
print("=" * 100)


In [ ]:
images_path = Path(IMAGES_DIR)

# Get all class folders
class_folders = sorted([d for d in images_path.iterdir() if d.is_dir()])

# Count total images for progress bar
total_images = 0
for class_folder in class_folders:
    image_files = list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png"))
    total_images += len(image_files)

print(f"Found {len(class_folders)} classes with {total_images:,} total images")

pbar = tqdm(total=total_images, desc="Processing images", unit="img")


for class_folder in class_folders:
    class_name = class_folder.name
    
    # Extract plant name from class name
    plant_name = extract_plant_name(class_name)
    print(f"\nClass: {class_name} - Plant: {plant_name}")